# Feature Engineering v2 — Hoàn chỉnh

## Những gì đã sửa so với v1
- ❌ Xoá `yoy_growth` (data leakage: dùng Revenue hiện tại để tính feature)
- ❌ Xoá `trend_multiplier` (dư thừa, dễ lệch khi extrapolate 548 ngày)
- ✅ Thêm `sessions_roll7` (đã tính sẵn ở v1 nhưng quên cho vào FEATURE_COLS)
- ✅ Thêm rolling stats (7, 14, 30 ngày) từ lag_365 làm backbone
- ✅ Thêm features từ `inventory.csv` — stockout rate, fill rate
- ✅ Thêm features từ `returns.csv` — return rate, refund rolling
- ✅ Thêm features từ `orders.csv` + `customers.csv` — new customer ratio, order volume
- ✅ Thêm payday effect (25th–5th Vietnamese salary cycle)
- ✅ Thêm promo_type, days_since_last_promo, promo_channel
- ✅ Xử lý đúng recursive lag: tách rõ lag available vs cần predicted values

## 1 — Imports & Config

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = './'
plt.rcParams['figure.figsize'] = (14, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print('✅ Ready')

## 2 — Load tất cả dữ liệu

In [ ]:
# Core
sales       = pd.read_csv(DATA_DIR + 'sales.csv',             parse_dates=['Date'])
submission  = pd.read_csv(DATA_DIR + 'sample_submission.csv', parse_dates=['Date'])
web_traffic = pd.read_csv(DATA_DIR + 'web_traffic.csv',       parse_dates=['date'])
promotions  = pd.read_csv(DATA_DIR + 'promotions.csv',        parse_dates=['start_date', 'end_date'])

# Nguồn dữ liệu mới
inventory   = pd.read_csv(DATA_DIR + 'inventory.csv',         parse_dates=['snapshot_date'])
returns     = pd.read_csv(DATA_DIR + 'returns.csv',           parse_dates=['return_date'])
orders      = pd.read_csv(DATA_DIR + 'orders.csv',            parse_dates=['order_date'])
customers   = pd.read_csv(DATA_DIR + 'customers.csv',         parse_dates=['signup_date'])

sales = sales.sort_values('Date').reset_index(drop=True)

TRAIN_END  = sales['Date'].max()           # 2022-12-31
PRED_START = submission['Date'].min()      # 2023-01-01
PRED_END   = submission['Date'].max()      # 2024-07-01

print(f'Train  : {sales.Date.min().date()} → {TRAIN_END.date()} ({len(sales)} ngày)')
print(f'Predict: {PRED_START.date()} → {PRED_END.date()} ({len(submission)} ngày)')

## 3 — Tạo full date range

In [ ]:
all_dates = pd.date_range(sales['Date'].min(), PRED_END, freq='D')
df = pd.DataFrame({'Date': all_dates})
df = df.merge(sales[['Date', 'Revenue', 'COGS']], on='Date', how='left')

print(f'Full dataframe : {len(df)} ngày')
print(f'Train rows     : {df.Revenue.notna().sum()}')
print(f'Predict rows   : {df.Revenue.isna().sum()}')

## 4 — Calendar features

In [ ]:
def add_calendar_features(df):
    df = df.copy()

    df['year']           = df['Date'].dt.year
    df['month']          = df['Date'].dt.month
    df['day']            = df['Date'].dt.day
    df['dayofweek']      = df['Date'].dt.dayofweek       # 0=Mon, 6=Sun
    df['dayofyear']      = df['Date'].dt.dayofyear
    df['weekofyear']     = df['Date'].dt.isocalendar().week.astype(int)
    df['quarter']        = df['Date'].dt.quarter
    df['is_weekend']     = (df['dayofweek'] >= 5).astype(int)
    df['is_month_end']   = df['Date'].dt.is_month_end.astype(int)
    df['is_month_start'] = df['Date'].dt.is_month_start.astype(int)
    df['is_quarter_end'] = df['Date'].dt.is_quarter_end.astype(int)
    df['dom_bin']        = pd.cut(df['day'], bins=[0, 10, 20, 31], labels=[0, 1, 2]).astype(int)

    # Linear trend (số ngày kể từ đầu — đủ để model học trend, không cần trend_multiplier)
    origin = df['Date'].min()
    df['trend'] = (df['Date'] - origin).dt.days

    # ── Payday effect (lương VN: ~25 hàng tháng)
    # Khoảng cách tới ngày 28 (trung điểm chu kỳ lương), dạng circular
    df['days_to_payday'] = df['day'].apply(lambda d: min(abs(d - 28), 28 - abs(d - 28) + 3))
    df['is_payday_window'] = ((df['day'] >= 25) | (df['day'] <= 5)).astype(int)

    return df

df = add_calendar_features(df)
print('✅ Calendar features:', [c for c in df.columns if c not in ['Date', 'Revenue', 'COGS']])

## 5 — Fourier seasonality

In [ ]:
def add_fourier_features(df, period, n_terms, col='dayofyear'):
    t = df[col].values
    for k in range(1, n_terms + 1):
        df[f'sin_{period}d_k{k}'] = np.sin(2 * np.pi * k * t / period)
        df[f'cos_{period}d_k{k}'] = np.cos(2 * np.pi * k * t / period)
    return df

# Yearly (3 cặp — giảm từ 4 để tránh overfit với 11 năm data)
df = add_fourier_features(df, period=365, n_terms=3, col='dayofyear')
# Weekly (3 cặp)
df = add_fourier_features(df, period=7,   n_terms=3, col='dayofweek')

fourier_cols = [c for c in df.columns if 'sin_' in c or 'cos_' in c]
print(f'✅ Fourier: {len(fourier_cols)} features')

## 6 — YoY lag backbone + rolling stats

Backbone: `lag_365` (same day last year) luôn available cho toàn bộ predict period.  
Rolling stats từ *quá khứ* (shift trước rồi mới rolling) — không leakage.

In [ ]:
def add_lag_and_rolling_features(df):
    df = df.copy()

    # ── Backbone lags (luôn available vì >= 365 ngày)
    df['rev_lag_365']  = df['Revenue'].shift(365)
    df['rev_lag_364']  = df['Revenue'].shift(364)   # align weekday tốt hơn
    df['rev_lag_728']  = df['Revenue'].shift(728)   # 2 năm trước
    df['cogs_lag_365'] = df['COGS'].shift(365)

    # ── Rolling stats từ lag_365 (smooth noise)
    # Dùng center=True trên window lag_365 để lấy ±3/7/15 ngày xung quanh cùng kỳ
    shifted = df['Revenue'].shift(365)
    df['rev_lag365_roll7']  = shifted.rolling(7,  center=True, min_periods=1).mean()
    df['rev_lag365_roll14'] = shifted.rolling(14, center=True, min_periods=1).mean()
    df['rev_lag365_roll30'] = shifted.rolling(30, center=True, min_periods=1).mean()
    df['rev_lag365_std7']   = shifted.rolling(7,  center=True, min_periods=1).std()
    df['rev_lag365_std30']  = shifted.rolling(30, center=True, min_periods=1).std()

    # ── Rolling stats ngắn hạn từ lag_365+offset
    # Ý nghĩa: "tuần này năm ngoái trông thế nào so với tháng ngoái"
    df['rev_mom_yoy'] = df['rev_lag365_roll7'] / (df['rev_lag365_roll30'] + 1e-6)  # momentum

    # ── Monthly avg năm ngoái (ít noise hơn daily)
    monthly_avg = (
        df[df['Revenue'].notna()]
        .groupby(['year', 'month'])['Revenue']
        .mean()
        .reset_index()
        .rename(columns={'Revenue': 'monthly_avg_rev'})
    )
    monthly_avg['year'] = monthly_avg['year'] + 1   # shift lên 1 năm
    df = df.merge(monthly_avg, on=['year', 'month'], how='left')

    # ── Log transform (giúp model học trend tốt hơn)
    df['log_rev_lag_365']  = np.log1p(df['rev_lag_365'])
    df['log_monthly_avg']  = np.log1p(df['monthly_avg_rev'])

    # ── NOTE: KHÔNG tính yoy_growth = Revenue/rev_lag_365 - 1
    # Lý do: dùng Revenue (target) để tính feature → data leakage trong training

    return df

df = add_lag_and_rolling_features(df)

# Verify tại ngày predict đầu và cuối
first_pred = df[df['Date'] == '2023-01-01'].iloc[0]
last_pred  = df[df['Date'] == '2024-07-01'].iloc[0]
print('=== 2023-01-01 ===')
print(f'rev_lag_365      = {first_pred.rev_lag_365:,.0f}  (Revenue 2022-01-01) ✅')
print(f'rev_lag365_roll7 = {first_pred.rev_lag365_roll7:,.0f}  ✅')
print()
print('=== 2024-07-01 ===')
print(f'rev_lag_365      = {last_pred.rev_lag_365}  (Revenue 2023-07-01 — cần predict trước!)')
print(f'rev_lag365_roll30= {last_pred.rev_lag365_roll30}  (từ 2023 — cần predict trước!)')
print()
print('⚠️  Các lag_365 cho nửa sau 2024 = NaN vì phụ thuộc vào 2023 predictions.')
print('   Xử lý: dùng monthly_avg_rev (từ tháng tương ứng năm 2022) làm fallback,')
print('   hoặc chạy recursive prediction (predict 2023 trước, inject làm lag cho 2024).')

## 6b — Year-level ratio adjustment cho lag_365

**Vấn đề xác nhận từ chẩn đoán:**
- 2019: mean_ratio = 0.614 → revenue 2019 chỉ bằng 61% của 2018
- `lag_365` raw dùng 2018 làm baseline → over-predict toàn bộ 2019 ~63%
- Tương tự với các năm có structural break (2017: 0.91, 2020: 0.93)

**Fix**: Tính `year_scale_ratio` = mean(năm hiện tại) / mean(năm trước)
dùng expanding window (không leakage) → nhân với `lag_365` để rescale về đúng level.

**Tại sao không leakage**: ratio được tính từ năm *trước* (đã biết), không dùng target năm hiện tại.

In [ ]:
def add_year_ratio_features(df):
    """
    Rescale lag_365 theo year-level ratio để sửa scale mismatch.
    
    Ý tưởng:
      - Tính annual mean revenue cho mỗi năm đã biết
      - year_scale_ratio[y] = mean(y-1) / mean(y-2)  ← expanding, không dùng year y
      - lag365_scaled = lag_365 * year_scale_ratio     ← baseline đúng level hơn
    """
    df = df.copy()

    # ── Tính annual mean từ train data (chỉ dùng năm đã có Revenue thực)
    annual_mean = (
        df[df['Revenue'].notna()]
        .groupby('year')['Revenue']
        .mean()
        .sort_index()
    )

    # ── Expanding YoY ratio: ratio[y] = mean[y] / mean[y-1]
    # Dùng expanding mean để smooth — tránh 1 năm bất thường kéo lệch
    yoy_ratio = annual_mean / annual_mean.shift(1)  # raw YoY

    # Expanding mean của ratio (càng về sau càng ổn định)
    expanding_ratio = yoy_ratio.expanding(min_periods=2).mean()

    print('Annual mean revenue:')
    for yr in annual_mean.index:
        raw   = yoy_ratio.get(yr, float('nan'))
        exp_r = expanding_ratio.get(yr, float('nan'))
        print(f'  {yr}: mean={annual_mean[yr]:>12,.0f}  yoy_ratio={raw:.4f}  expanding_ratio={exp_r:.4f}')

    # ── Map ratio vào df: mỗi row dùng expanding_ratio của năm TRƯỚC đó
    # year y dùng expanding_ratio[y-1] → không leakage
    ratio_lag1 = expanding_ratio.shift(1)  # ratio của năm trước
    ratio_map  = ratio_lag1.to_dict()

    df['year_scale_ratio'] = df['year'].map(ratio_map)

    # Fill năm đầu tiên (chưa có ratio) bằng 1.0
    df['year_scale_ratio'] = df['year_scale_ratio'].fillna(1.0)

    # ── Rescaled lag features
    # lag365_scaled: baseline đã điều chỉnh theo trend năm
    df['rev_lag365_scaled']      = df['rev_lag_365']        * df['year_scale_ratio']
    df['rev_lag365_roll7_scaled']= df['rev_lag365_roll7']   * df['year_scale_ratio']
    df['rev_lag365_roll30_scaled']= df['rev_lag365_roll30'] * df['year_scale_ratio']

    # ── Log của scaled lag (smooth hơn cho LGBM)
    df['log_rev_lag365_scaled']  = np.log1p(df['rev_lag365_scaled'])

    # ── Deviation: lag_365 raw vs scaled → signal về structural break
    # Nếu ratio << 1 → năm này expected thấp hơn → feature mang thông tin về break
    df['lag365_scale_deviation'] = df['rev_lag_365'] - df['rev_lag365_scaled']

    return df, ratio_map


df_train_all = pd.concat([
    train_df.assign(split='train'),
    pred_df.assign(split='pred')
]).sort_values('Date').reset_index(drop=True)

df_train_all, ratio_map = add_year_ratio_features(df_train_all)

# Split lại
train_df = df_train_all[df_train_all['split']=='train'].drop(columns='split').reset_index(drop=True)
pred_df  = df_train_all[df_train_all['split']=='pred'].drop(columns='split').reset_index(drop=True)

print()
# Verify 2019: lag365_scaled phải gần với actual 2019 hơn lag_365 raw
yr2019 = train_df[train_df['year']==2019]
print(f'=== Verify 2019 ===')
print(f'Actual mean        : {yr2019.Revenue.mean():>12,.0f}')
print(f'lag_365 mean (raw) : {yr2019.rev_lag_365.mean():>12,.0f}  (over-predict baseline)')
print(f'lag365_scaled mean : {yr2019.rev_lag365_scaled.mean():>12,.0f}  (adjusted baseline)')
print(f'year_scale_ratio   : {yr2019.year_scale_ratio.iloc[0]:.4f}')


## 7 — Vietnam holidays

In [ ]:
def add_vietnam_holidays(df):
    df = df.copy()

    # Tết Nguyên Đán
    tet_dates = pd.to_datetime([
        '2013-02-10', '2014-01-31', '2015-02-19', '2016-02-08',
        '2017-01-28', '2018-02-16', '2019-02-05', '2020-01-25',
        '2021-02-12', '2022-02-01', '2023-01-22', '2024-02-10'
    ])

    # Ngày lễ cố định
    fixed_holidays = []
    for year in range(2012, 2025):
        fixed_holidays.extend([
            f'{year}-01-01',  # Tết Dương lịch
            f'{year}-04-10',  # Giỗ Tổ Hùng Vương (thêm mới)
            f'{year}-04-30',  # Giải phóng Miền Nam
            f'{year}-05-01',  # Quốc tế Lao động
            f'{year}-09-02',  # Quốc khánh
        ])
    fixed_holidays = pd.to_datetime(fixed_holidays)

    # Khoảng cách tới Tết (continuous)
    df['days_to_tet'] = df['Date'].apply(
        lambda d: min(abs((d - t).days) for t in tet_dates)
    )

    # Zones xung quanh Tết
    df['pre_tet_7d']  = (df['days_to_tet'] <= 7).astype(int)
    df['pre_tet_14d'] = ((df['days_to_tet'] <= 14) & (df['days_to_tet'] > 7)).astype(int)
    df['post_tet_7d'] = df['Date'].apply(
        lambda d: any(0 < (d - t).days <= 7 for t in tet_dates)
    ).astype(int)

    df['is_fixed_holiday'] = df['Date'].isin(fixed_holidays).astype(int)
    df['is_tet_holiday']   = df['Date'].isin(tet_dates).astype(int)

    # E-commerce events
    df['is_black_friday_month'] = ((df['month'] == 11) & (df['day'] >= 20)).astype(int)
    df['is_1111'] = ((df['month'] == 11) & (df['day'] == 11)).astype(int)
    df['is_1212'] = ((df['month'] == 12) & (df['day'] == 12)).astype(int)
    df['is_0808'] = ((df['month'] == 8)  & (df['day'] == 8)).astype(int)   # thêm 8/8

    return df

df = add_vietnam_holidays(df)
print('✅ Holiday features:', [c for c in df.columns if any(x in c for x in ['tet','holiday','11','12','payday'])])

## 8 — Web traffic features

In [ ]:
def add_traffic_features(df, web_traffic):
    df = df.copy()

    daily_traffic = (
        web_traffic
        .groupby('date')
        .agg(
            sessions         = ('sessions',              'sum'),
            unique_visitors  = ('unique_visitors',       'sum'),
            bounce_rate      = ('bounce_rate',           'mean'),
            avg_session_dur  = ('avg_session_duration_sec', 'mean')
        )
        .reset_index()
        .rename(columns={'date': 'Date'})
    )

    df = df.merge(daily_traffic, on='Date', how='left')

    # ── sessions_roll7: shift(1) trước khi rolling → không leakage cho next-day forecast
    # (đã tính ở v1 nhưng bị bỏ sót khỏi FEATURE_COLS — đã sửa)
    df['sessions_roll7']  = df['sessions'].shift(1).rolling(7,  min_periods=1).mean()
    df['sessions_roll14'] = df['sessions'].shift(1).rolling(14, min_periods=1).mean()

    # ── YoY lag traffic (luôn available)
    df['sessions_lag_365']  = df['sessions'].shift(365)
    df['visitors_lag_365']  = df['unique_visitors'].shift(365)

    # ── Traffic momentum: tuần này so với trung bình 30 ngày gần nhất (YoY)
    df['traffic_momentum'] = (
        df['sessions'].shift(365).rolling(7, min_periods=1).mean() /
        (df['sessions'].shift(365).rolling(30, min_periods=1).mean() + 1e-6)
    )

    return df

df = add_traffic_features(df, web_traffic)
print('✅ Traffic features added')

# Correlation check
train_tmp = df[df['Revenue'].notna()]
print('\nCorrelation với Revenue:')
for col in ['sessions', 'sessions_roll7', 'sessions_lag_365', 'traffic_momentum']:
    if col in train_tmp.columns:
        c = train_tmp['Revenue'].corr(train_tmp[col])
        print(f'  {col:<25}: r={c:.4f}')

## 9 — Promotion features (mở rộng)

In [ ]:
def add_promotion_features(df, promotions):
    df = df.copy()
    df['n_active_promos']       = 0
    df['max_promo_discount']    = 0.0
    df['has_stackable_promo']   = 0
    df['promo_is_percentage']   = 0   # mới: loại promo (% vs fixed)
    df['promo_online_channel']  = 0   # mới: kênh online vs offline

    for _, promo in promotions.iterrows():
        mask = (df['Date'] >= promo['start_date']) & (df['Date'] <= promo['end_date'])
        df.loc[mask, 'n_active_promos']    += 1
        df.loc[mask, 'max_promo_discount']  = df.loc[mask, 'max_promo_discount'].clip(
            lower=promo.get('discount_value', 0)
        )
        if promo.get('stackable_flag', 0):
            df.loc[mask, 'has_stackable_promo'] = 1
        if str(promo.get('promo_type', '')).lower() == 'percentage':
            df.loc[mask, 'promo_is_percentage'] = 1
        if str(promo.get('promo_channel', '')).lower() in ['online', 'email']:
            df.loc[mask, 'promo_online_channel'] = 1

    # ── Days since last promo ended (post-promo dip effect)
    # Sau promo kết thúc, khách đã mua bị satiated → Revenue giảm
    df['days_since_promo_end'] = np.nan
    last_end = pd.NaT
    for idx, row in df.iterrows():
        # Kiểm tra xem ngày hôm nay có promo kết thúc không
        ended_today = promotions[promotions['end_date'] == row['Date']]
        if len(ended_today) > 0:
            last_end = row['Date']
        if pd.notna(last_end) and row['n_active_promos'] == 0:
            df.loc[idx, 'days_since_promo_end'] = (row['Date'] - last_end).days

    # Fill NaN (chưa có promo nào kết thúc) bằng 999 (xa vô cực)
    df['days_since_promo_end'] = df['days_since_promo_end'].fillna(999).clip(upper=60)

    return df

df = add_promotion_features(df, promotions)
print('✅ Promotion features added')
train_tmp = df[df['Revenue'].notna()]
print(f'Ngày có promo    : {(train_tmp.n_active_promos > 0).sum()}')
print(f'Avg rev có promo : {train_tmp[train_tmp.n_active_promos>0].Revenue.mean():,.0f}')
print(f'Avg rev không    : {train_tmp[train_tmp.n_active_promos==0].Revenue.mean():,.0f}')

## 10 — Inventory features (MỚI)

`inventory.csv` là snapshot cuối tháng. Ta aggregate thành daily fill_rate và stockout_rate,
rồi forward-fill trong tháng đó.

In [ ]:
def add_inventory_features(df, inventory):
    df = df.copy()

    # Aggregate theo tháng: trung bình trên toàn bộ sản phẩm
    monthly_inv = (
        inventory
        .groupby(['year', 'month'])
        .agg(
            avg_fill_rate      = ('fill_rate',      'mean'),
            stockout_rate      = ('stockout_flag',  'mean'),   # tỷ lệ SKU bị stockout
            avg_days_of_supply = ('days_of_supply', 'mean'),
            avg_sell_through   = ('sell_through_rate', 'mean')
        )
        .reset_index()
    )

    # Merge vào df theo year-month
    df = df.merge(monthly_inv, on=['year', 'month'], how='left')

    # ── Inventory YoY lag: dùng năm ngoái cùng tháng (luôn available)
    monthly_inv_lag = monthly_inv.copy()
    monthly_inv_lag['year'] = monthly_inv_lag['year'] + 1
    monthly_inv_lag = monthly_inv_lag.rename(columns={
        'avg_fill_rate'      : 'fill_rate_lag_1y',
        'stockout_rate'      : 'stockout_rate_lag_1y',
        'avg_days_of_supply' : 'days_supply_lag_1y',
        'avg_sell_through'   : 'sell_through_lag_1y'
    })
    df = df.merge(monthly_inv_lag, on=['year', 'month'], how='left')

    return df

df = add_inventory_features(df, inventory)
print('✅ Inventory features added')

# Verify correlation
train_tmp = df[df['Revenue'].notna()]
print('\nCorrelation với Revenue:')
for col in ['avg_fill_rate', 'stockout_rate', 'avg_days_of_supply', 'fill_rate_lag_1y', 'stockout_rate_lag_1y']:
    if col in train_tmp.columns:
        c = train_tmp['Revenue'].corr(train_tmp[col])
        print(f'  {col:<30}: r={c:.4f}')

## 11 — Returns features (MỚI)

Return rate cao sau các đợt sale lớn làm Revenue gộp bị inflate.
Đưa return signal vào giúp model nhận biết pattern này.

In [ ]:
def add_return_features(df, returns):
    df = df.copy()

    # Aggregate returns theo ngày
    daily_ret = (
        returns
        .groupby('return_date')
        .agg(
            daily_refund_amount = ('refund_amount',   'sum'),
            daily_return_qty    = ('return_quantity', 'sum'),
            n_returns           = ('return_id',       'count')
        )
        .reset_index()
        .rename(columns={'return_date': 'Date'})
    )

    df = df.merge(daily_ret, on='Date', how='left')
    df[['daily_refund_amount', 'daily_return_qty', 'n_returns']] = \
        df[['daily_refund_amount', 'daily_return_qty', 'n_returns']].fillna(0)

    # ── Rolling return metrics (shift(1) trước — không dùng ngày hiện tại)
    df['return_amount_roll7']  = df['daily_refund_amount'].shift(1).rolling(7,  min_periods=1).mean()
    df['return_amount_roll30'] = df['daily_refund_amount'].shift(1).rolling(30, min_periods=1).mean()
    df['return_qty_roll7']     = df['daily_return_qty'].shift(1).rolling(7,    min_periods=1).mean()

    # ── Return rate so với Revenue lag (proxy net Revenue)
    # shift(1) để không dùng Revenue ngày hiện tại
    df['return_rate_roll7'] = (
        df['daily_refund_amount'].shift(1).rolling(7, min_periods=1).sum() /
        (df['Revenue'].shift(1).rolling(7, min_periods=1).sum() + 1e-6)
    )

    # ── YoY lag returns (luôn available)
    df['return_amount_lag_365'] = df['daily_refund_amount'].shift(365)

    return df

df = add_return_features(df, returns)
print('✅ Returns features added')

train_tmp = df[df['Revenue'].notna()]
print('\nCorrelation với Revenue:')
for col in ['return_amount_roll7', 'return_rate_roll7', 'return_amount_lag_365']:
    c = train_tmp['Revenue'].corr(train_tmp[col])
    print(f'  {col:<30}: r={c:.4f}')

## 12 — Customer behavior features (MỚI)

New vs returning customer ratio là leading indicator: new customers tăng vọt trong promo,
tạo sóng repeat-purchase Revenue 14–30 ngày sau.

In [ ]:
def add_customer_features(df, orders, customers):
    df = df.copy()

    # Join orders với customers để lấy signup_date
    ord_cust = orders.merge(
        customers[['customer_id', 'signup_date']],
        on='customer_id', how='left'
    )

    # Định nghĩa "new customer": đặt hàng trong vòng 30 ngày kể từ ngày đăng ký
    ord_cust['is_new_customer'] = (
        (ord_cust['order_date'] - ord_cust['signup_date']).dt.days <= 30
    ).astype(int)

    # Aggregate theo ngày
    daily_orders = (
        ord_cust
        .groupby('order_date')
        .agg(
            total_orders         = ('order_id',       'count'),
            new_customer_orders  = ('is_new_customer','sum'),
            unique_customers     = ('customer_id',    'nunique')
        )
        .reset_index()
        .rename(columns={'order_date': 'Date'})
    )
    daily_orders['new_customer_ratio'] = (
        daily_orders['new_customer_orders'] / daily_orders['total_orders']
    )

    df = df.merge(daily_orders, on='Date', how='left')

    # ── Rolling order volume (shift(1) — không leakage)
    df['orders_roll7']  = df['total_orders'].shift(1).rolling(7,  min_periods=1).mean()
    df['orders_roll30'] = df['total_orders'].shift(1).rolling(30, min_periods=1).mean()

    # ── New customer ratio rolling
    df['new_cust_ratio_roll7']  = df['new_customer_ratio'].shift(1).rolling(7,  min_periods=1).mean()
    df['new_cust_ratio_roll30'] = df['new_customer_ratio'].shift(1).rolling(30, min_periods=1).mean()

    # ── Leading indicator: new customer spike → repeat purchase 14-30 ngày sau
    # Dùng lag 14 và 21 ngày của new_customer_ratio
    df['new_cust_ratio_lag14'] = df['new_customer_ratio'].shift(14)
    df['new_cust_ratio_lag21'] = df['new_customer_ratio'].shift(21)

    # ── YoY order volume lag (luôn available)
    df['orders_lag_365'] = df['total_orders'].shift(365)

    return df

df = add_customer_features(df, orders, customers)
print('✅ Customer features added')

train_tmp = df[df['Revenue'].notna()]
print('\nCorrelation với Revenue:')
for col in ['total_orders', 'orders_roll7', 'new_customer_ratio',
            'new_cust_ratio_lag14', 'orders_lag_365']:
    if col in train_tmp.columns:
        c = train_tmp['Revenue'].corr(train_tmp[col])
        print(f'  {col:<30}: r={c:.4f}')

## 13 — Tổng hợp FEATURE_COLS

In [ ]:
FEATURE_COLS = [

    # ── CALENDAR
    'year', 'month', 'day', 'dayofweek', 'dayofyear',
    'weekofyear', 'quarter', 'is_weekend',
    'is_month_end', 'is_month_start', 'is_quarter_end', 'dom_bin',
    'trend',
    'days_to_payday', 'is_payday_window',

    # ── FOURIER SEASONALITY
    'sin_365d_k1', 'cos_365d_k1',
    'sin_365d_k2', 'cos_365d_k2',
    'sin_365d_k3', 'cos_365d_k3',
    'sin_7d_k1',  'cos_7d_k1',
    'sin_7d_k2',  'cos_7d_k2',
    'sin_7d_k3',  'cos_7d_k3',

    # ── YoY BACKBONE LAGS (luôn available cho predict period)
    'rev_lag_365', 'rev_lag_364', 'rev_lag_728',
    'cogs_lag_365',
    'log_rev_lag_365', 'log_monthly_avg', 'monthly_avg_rev',

    # ── YEAR-LEVEL RATIO ADJUSTMENT (FIX: scale mismatch 2019, structural breaks)
    'year_scale_ratio',
    'rev_lag365_scaled', 'rev_lag365_roll7_scaled', 'rev_lag365_roll30_scaled',
    'log_rev_lag365_scaled', 'lag365_scale_deviation',

    # ── ROLLING STATS từ lag_365 (backbone + context)
    'rev_lag365_roll7', 'rev_lag365_roll14', 'rev_lag365_roll30',
    'rev_lag365_std7',  'rev_lag365_std30',
    'rev_mom_yoy',

    # ── HOLIDAY
    'days_to_tet', 'pre_tet_7d', 'pre_tet_14d', 'post_tet_7d',
    'is_fixed_holiday', 'is_tet_holiday',
    'is_black_friday_month', 'is_1111', 'is_1212', 'is_0808',

    # ── WEB TRAFFIC
    'sessions_roll7', 'sessions_roll14',        # safe: shift(1) trước rolling
    'sessions_lag_365', 'visitors_lag_365',     # YoY lags
    'traffic_momentum',

    # ── PROMOTIONS
    'n_active_promos', 'max_promo_discount',
    'has_stackable_promo', 'promo_is_percentage', 'promo_online_channel',
    'days_since_promo_end',

    # ── INVENTORY (MỚI)
    'avg_fill_rate', 'stockout_rate', 'avg_days_of_supply', 'avg_sell_through',
    'fill_rate_lag_1y', 'stockout_rate_lag_1y',

    # ── RETURNS (MỚI)
    'return_amount_roll7', 'return_amount_roll30',
    'return_qty_roll7', 'return_rate_roll7',
    'return_amount_lag_365',

    # ── CUSTOMER BEHAVIOR (MỚI)
    'orders_roll7', 'orders_roll30',
    'new_cust_ratio_roll7', 'new_cust_ratio_roll30',
    'new_cust_ratio_lag14', 'new_cust_ratio_lag21',
    'orders_lag_365',
]

# Lọc chỉ lấy cols tồn tại trong df
FEATURE_COLS = [c for c in FEATURE_COLS if c in df.columns]

print(f'Tổng số features: {len(FEATURE_COLS)}')
print()

# Tách train / predict set
train_df = df[df['Revenue'].notna()].copy()
train_df = train_df.dropna(subset=['rev_lag_365'])   # cần ít nhất 365 ngày
pred_df  = df[df['Date'] >= PRED_START].copy()

print(f'Train set  : {len(train_df)} rows')
print(f'Predict set: {len(pred_df)} rows')

# Check nulls
null_train = train_df[FEATURE_COLS].isnull().sum()
null_pred  = pred_df[FEATURE_COLS].isnull().sum()
print('\nNull trong train set:')
print(null_train[null_train > 0] if null_train.sum() > 0 else '  None ✅')
print('\nNull trong predict set (trước fill):')
print(null_pred[null_pred > 0] if null_pred.sum() > 0 else '  None ✅')

## 14 — Fill NaN & xử lý recursive lag

Các lag_365 cho nửa sau 2024 sẽ NaN vì phụ thuộc vào predictions 2023.
Strategy: fallback về `monthly_avg_rev` (lag 1 năm theo tháng) cho các NaN đó.

In [ ]:
# ── Recursive lag fallback cho predict set
# Các cột lag_365 / roll từ lag_365 sẽ NaN với ngày > 2023-12-31
# Fallback: monthly_avg_rev * seasonal ratio (giống logic baseline)
lag_cols_with_fallback = [
    'rev_lag_365', 'rev_lag_364', 'rev_lag_728',
    'rev_lag365_roll7', 'rev_lag365_roll14', 'rev_lag365_roll30',
    'log_rev_lag_365'
]

for col in lag_cols_with_fallback:
    if col in pred_df.columns:
        # Fill NaN bằng monthly_avg_rev của tháng đó (lag 1 năm theo tháng)
        null_mask = pred_df[col].isnull()
        if null_mask.sum() > 0:
            if col == 'log_rev_lag_365':
                pred_df.loc[null_mask, col] = np.log1p(pred_df.loc[null_mask, 'monthly_avg_rev'])
            else:
                pred_df.loc[null_mask, col] = pred_df.loc[null_mask, 'monthly_avg_rev']
            print(f'  Filled {null_mask.sum()} NaN in {col} with monthly_avg_rev fallback')

# ── Fill các null còn lại bằng median của train set
for col in FEATURE_COLS:
    if train_df[col].isnull().any() or pred_df[col].isnull().any():
        med = train_df[col].median()
        train_df[col] = train_df[col].fillna(med)
        pred_df[col]  = pred_df[col].fillna(med)

print(f'\nTrain nulls còn lại : {train_df[FEATURE_COLS].isnull().sum().sum()} ✅')
print(f'Predict nulls còn lại: {pred_df[FEATURE_COLS].isnull().sum().sum()} ✅')

## 15 — Correlation overview & Feature importance preview

In [ ]:
corr = train_df[FEATURE_COLS + ['Revenue']].corr()['Revenue'].drop('Revenue')
corr_sorted = corr.abs().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 14))
colors = ['steelblue' if corr[c] > 0 else 'salmon' for c in corr_sorted.index]
ax.barh(corr_sorted.index, corr_sorted.values, color=colors)
ax.set_title('Feature Correlation với Revenue (|r|) — v2')
ax.set_xlabel('|Pearson r|')
plt.tight_layout()
plt.show()

print('\nTop 15 features tương quan nhất:')
for feat, val in corr_sorted.head(15).items():
    direction = '+' if corr[feat] > 0 else '-'
    print(f'  {feat:<35}: {direction}{val:.4f}')

## 16 — Export

In [ ]:
train_out = train_df[['Date', 'Revenue', 'COGS'] + FEATURE_COLS].copy()
pred_out  = pred_df[['Date'] + FEATURE_COLS].copy()

train_out.to_csv('features_train_v2.csv', index=False)
pred_out.to_csv('features_predict_v2.csv', index=False)

# COGS ratio để predict COGS từ Revenue
cogs_ratio = train_df['Revenue'].sum() / train_df['COGS'].sum()

import json
meta = {
    'version'      : 'v2',
    'feature_cols' : FEATURE_COLS,
    'n_features'   : len(FEATURE_COLS),
    'cogs_ratio'   : cogs_ratio,
    'n_train'      : len(train_out),
    'n_predict'    : len(pred_out),
    'train_start'  : str(train_out.Date.min().date()),
    'train_end'    : str(train_out.Date.max().date()),
    'pred_start'   : str(pred_out.Date.min().date()),
    'pred_end'     : str(pred_out.Date.max().date()),
    'fixes_applied': [
        'Removed yoy_growth (data leakage)',
        'Removed trend_multiplier (redundant)',
        'Added sessions_roll7 (was computed but missing from FEATURE_COLS)',
        'Added rolling stats 7/14/30 from lag_365',
        'Added inventory features (stockout_rate, fill_rate)',
        'Added return features (return_rate_roll7, return_amount_roll30)',
        'Added customer behavior (new_cust_ratio, orders_roll7)',
        'Added payday effect, promo_type, days_since_promo_end',
        'Recursive lag fallback for 2024 predictions',
    ]
}
with open('feature_meta_v2.json', 'w') as f:
    json.dump(meta, f, indent=2)

print(f'COGS ratio (Revenue/COGS): {cogs_ratio:.6f}')
print()
print('✅ Đã lưu:')
print(f'  features_train_v2.csv    — {len(train_out)} rows × {len(FEATURE_COLS)+3} cols')
print(f'  features_predict_v2.csv  — {len(pred_out)} rows × {len(FEATURE_COLS)+1} cols')
print(f'  feature_meta_v2.json     — metadata')
print()
print('=== TIẾP THEO: Train model ===')
print('  X_train = train[FEATURE_COLS]')
print('  y_train = np.log1p(train["Revenue"])   # log transform giảm ảnh hưởng outlier')
print('  Model  : LightGBM với TimeSeriesSplit(n_splits=5, gap=90)')
print('  Predict COGS: pred["COGS"] = pred["Revenue"] / cogs_ratio')